# BigCodeBench evaluator for EvaLoop (Colab, free tier)

This notebook runs the BigCodeBench evaluator on this Colab VM and exposes it as a temporary Gradio share URL. EvaLoop (`evaloop evaluate --dataset bigcodebench`) sends each cycle's generated code here and reads back the test results. **Model-generated code runs only on this VM**, never on the machine that runs EvaLoop.

**Before you start:**

- **Use a fresh, dedicated runtime:** Use a CPU runtime on the free tier; a GPU is not needed.
- **Don't connect anything to it:** no Google Drive mount, no Colab secrets, no sign-ins. The evaluator executes untrusted code, and anyone who has the share URL can send code to it until the runtime stops.
- **Run the cells in order.** Installing takes about 25 minutes; the ground-truth preload of `hard` takes roughly another 10 minutes on 2 vCPUs.
- **Treat the share URL as a secret.** Pass it to EvaLoop with the `EVALOOP_GRADIO_ENDPOINT` environment variable. Don't paste it into files or issues, and don't save this notebook with its outputs.
- **Stop the runtime when the run ends**. The Gradio share URL immediately stop working.

**Pinned versions** (they must match EvaLoop's `pyproject.toml` extra and the README):

| | Version |
|---|---|
| Python (evaluator environment) | 3.10, as in the official evaluator image |
| `bigcodebench` (dataset `v0.1.4`) | 0.2.5 |
| BigCodeBench evaluation requirements | `Requirements/requirements-eval.txt` at tag `v0.2.5` |
| `gradio` (requires `gradio_client` 2.7.2, EvaLoop's pin) | 6.29.1 |
| `datasets` | 4.5.0 |

**Attribution:** the evaluator app in step 4 is adapted from the [`bigcode/bigcodebench-evaluator`](https://huggingface.co/spaces/bigcode/bigcodebench-evaluator) Space (`app.py` at commit `8653fc7f`), licensed under the [Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0). The changes are listed at the top of that cell. The environment follows the Space's `Docker/Gradio.Dockerfile` at BigCodeBench `v0.2.5`.

## 1. Check the runtime

Prints the CPU count and memory. EvaLoop's evaluator limits are fixed (2 workers; per test process a 1 s minimum time limit, 4 GB address-space and data limits and a 10 MB stack), sized for a free CPU runtime. `--bcb_parallel` can't exceed the CPU count (the app rejects larger values).

**Task set:** EvaLoop runs BigCodeBench-Hard (148 tasks), sized for a free-tier session.

In [ ]:
import os
import shutil
import subprocess

cpus = os.cpu_count() or 1
with open("/proc/meminfo") as meminfo:
    mem_kb = {line.split(":")[0]: int(line.split()[1]) for line in meminfo if line.split()[1].isdigit()}
ram_mb = mem_kb["MemTotal"] // 1024
disk_free_gb = shutil.disk_usage("/").free / 1e9

print(f"CPUs: {cpus}   RAM: {ram_mb / 1024:.1f} GB   free disk: {disk_free_gb:.0f} GB")

if shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
    print("WARNING: this runtime has a GPU. The protocol uses a free CPU runtime: switch with Runtime -> Change runtime type.")
if os.path.ismount("/content/drive"):
    print("WARNING: Google Drive is mounted. Use a fresh runtime without Drive: generated code runs here.")

# EvaLoop's fixed limits: 2 workers, each test process capped at 4 GB address space and data.
if cpus < 2:
    print("WARNING: fewer than 2 CPUs. Run EvaLoop with --bcb_parallel 1.")
if ram_mb < 12 * 1024:
    print("WARNING: less than 12 GB RAM. Two 4 GB test processes plus the app may not fit; use a standard free CPU runtime.")

## 2. Settings

In [ ]:
PYTHON_VERSION = "3.10"
BIGCODEBENCH_VERSION = "0.2.5"
BIGCODEBENCH_TAG = "v0.2.5"
GRADIO_VERSION = "6.29.1"
DATASETS_VERSION = "4.5.0"
EVAL_REQUIREMENTS_URL = (
    f"https://raw.githubusercontent.com/bigcode-project/bigcodebench/{BIGCODEBENCH_TAG}/Requirements/requirements-eval.txt"
)

ENV_DIR = "/opt/bcb-env"         # Python 3.10 environment of the evaluator
PYTHON_DIR = "/opt/uv-python"    # where uv puts Python 3.10 (not root's home, which RUN_USER can't read)
APP_PATH = "/opt/bcb-app/app.py"  # the evaluator app (step 4)
RUN_USER = "bcbeval"             # unprivileged user that runs the app, as in the official image
LOG_PATH = "/content/evaluator.log"

# The evaluator runs with a clean environment, not the notebook's: Colab sets variables such as
# MPLBACKEND=module://matplotlib_inline.backend_inline, which made every plotting task fail.
EVALUATOR_ENV = [
    f"HOME=/home/{RUN_USER}", f"PATH={ENV_DIR}/bin:/usr/local/bin:/usr/bin:/bin",
    "LANG=C.UTF-8", "MPLBACKEND=Agg", "PYTHONUNBUFFERED=1",
]

## 3. Install (~25 minutes)

Mirrors the official evaluator image:

1. System packages from the image.
2. A separate **Python 3.10** environment, created with `uv`. Colab's own Python is newer, and the evaluation requirements (e.g. `tensorflow==2.11.0`, `numpy==1.21.2`, `scipy==1.7.2`) only install on 3.10.
3. Pinned `gradio`, `bigcodebench` (without its generation backends, as in the image) and its runtime dependencies, and `datasets`.
4. The evaluation requirements, installed last so that their pins win, as in the image. Unlike the image, this uses a normal install, not `pip install -I`: `-I` writes the pinned versions over newer ones without removing them, which leaves a broken mix (e.g. files from numpy 2.x and 1.21 together).

The last steps print `pip check`. Conflicts it reports between the evaluation pins and newer packages (e.g. `requests`) are expected, as in the official image; the import check at the end must pass. The full package list is saved to `/content/evaluator-requirements.txt`.

In [ ]:
import os
import subprocess
import time

def run(command):
    started = time.time()
    print(f"$ {command}", flush=True)
    subprocess.run(command, shell=True, check=True)
    print(f"  done in {time.time() - started:.0f}s", flush=True)

PIP = f"{ENV_DIR}/bin/python -m pip"

run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "git g++ python3-tk zip unzip procps r-base libgdal-dev libfreetype6-dev libpng-dev pkg-config > /dev/null")
run("pip install -q uv")
run(f"UV_PYTHON_INSTALL_DIR={PYTHON_DIR} uv venv --seed --python {PYTHON_VERSION} {ENV_DIR}")
run(f"{PIP} install -q --upgrade pip")
run(f"{PIP} install -q gradio=={GRADIO_VERSION}")
run(f"{PIP} install -q --no-deps bigcodebench=={BIGCODEBENCH_VERSION}")
run(f"{PIP} install -q 'appdirs>=1.4.4' 'fire>=0.6.0' 'multipledispatch>=0.6.0' 'pqdm>=0.2.0' "
    "'tempdir>=0.7.1' 'termcolor>=2.0.0' 'tqdm>=4.56.0' 'wget>=3.2' rich")
run(f"{PIP} install -q datasets=={DATASETS_VERSION}")
run(f"{PIP} install -q --timeout 2000 -r {EVAL_REQUIREMENTS_URL}")
run(f"{PIP} install -q setuptools==69.5.1")  # provides pkg_resources, which a few tasks import

run(f"{PIP} freeze > /content/evaluator-requirements.txt")
subprocess.run(f"{PIP} check", shell=True)  # informational, see above

check = (
    "import sys, gradio, bigcodebench, bigcodebench.eval, bigcodebench.gen.util, datasets, numpy, pyarrow; "
    "print('python', sys.version.split()[0], '| gradio', gradio.__version__, '| bigcodebench', bigcodebench.__version__, "
    "'| datasets', datasets.__version__, '| numpy', numpy.__version__, '| pyarrow', pyarrow.__version__)"
)
if subprocess.run(["id", RUN_USER], capture_output=True).returncode != 0:
    run(f"useradd --create-home --shell /bin/bash {RUN_USER}")
run(f"chmod -R a+rX {PYTHON_DIR} {ENV_DIR}")

# Checked as RUN_USER, the user that runs the app in step 5.
output = subprocess.run(["runuser", "-u", RUN_USER, "--", "env", "-i", *EVALUATOR_ENV, f"{ENV_DIR}/bin/python", "-c", check],
                        capture_output=True, text=True, cwd="/tmp")
print(output.stdout or output.stderr)
assert output.returncode == 0, "The evaluator environment failed its import check; see the error above."
assert f"gradio {GRADIO_VERSION}" in output.stdout, "Unexpected gradio version"
assert f"bigcodebench {BIGCODEBENCH_VERSION}" in output.stdout, "Unexpected bigcodebench version"
assert "numpy 1.21.2" in output.stdout, "numpy is not the version the evaluation requirements pin"

os.makedirs(os.path.dirname(APP_PATH), exist_ok=True)
print("Install complete.")

## 4. Write the evaluator app

The `/predict` endpoint keeps the Space's exact parameters, which EvaLoop's tester calls. See the header of the cell for what changed and why.

In [ ]:
%%writefile /opt/bcb-app/app.py
# BigCodeBench evaluator for EvaLoop, adapted from the bigcode/bigcodebench-evaluator Space
# (https://huggingface.co/spaces/bigcode/bigcodebench-evaluator, app.py at commit
# 8653fc7f5e4f8c268d84e24d48f3548648a267d0), licensed under the Apache License 2.0
# (https://www.apache.org/licenses/LICENSE-2.0).
#
# Changes from the original (the /predict endpoint and its parameters are unchanged):
# - Removed the hourly HfApi.restart_space scheduler (and the HfApi / APScheduler imports).
# - Replaced the silent infinite ground-truth preload loop with bounded retries that print
#   errors, for the subsets given on the command line.
# - Set api_name="predict" explicitly: Gradio 6 would otherwise name the endpoint after the
#   function (/evaluate).
# - Converted NumPy values in the outputs to plain Python numbers for the JSON outputs.
# - Cast the worker count and memory limits to int.
# - Log one line per request (subset, sample count, mode, duration).
# - Launch with share=True and command-line options.

import argparse
import gc
import json
import logging
import multiprocessing
import os
import pickle
import sys
import time
import traceback
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from typing import Any, Dict, List, Tuple

import gradio as gr
import numpy as np
from bigcodebench.data import get_bigcodebench, get_bigcodebench_hash, load_solutions
from bigcodebench.data.utils import CACHE_DIR
from bigcodebench.eval import PASS, estimate_pass_at_k, untrusted_check
from bigcodebench.gen.util import trusted_check

Result = Tuple[str, List[bool]]

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", stream=sys.stdout)
log = logging.getLogger("evaluator")


def get_groundtruth(n_workers, problems, hashcode, check_gt_only, max_as_limit, max_data_limit, max_stack_limit, min_time_limit):
    cache_file = os.path.join(CACHE_DIR, f"{hashcode}.pkl")
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    os.makedirs(CACHE_DIR, exist_ok=True)

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = []
        n_samples = 0
        expected_time = dict()

        for problem in problems.values():
            args = (
                problem["complete_prompt"] + "\n" + problem["canonical_solution"],
                problem["test"],
                problem["task_id"],
                max_as_limit,
                max_data_limit,
                max_stack_limit,
                min_time_limit,
            )

            futures.append(executor.submit(trusted_check, *args))
            n_samples += 1

        for future in as_completed(futures):
            result = future.result()
            expected_time[result["task_id"]] = result["time"]

    if any(expected_time.values()):
        with open(cache_file, "wb") as f:
            pickle.dump(expected_time, f)

    return expected_time


def check_correctness(
    completion_id: int,
    problem: Dict[str, Any],
    solution: str,
    max_as_limit: float,
    max_data_limit: float,
    max_stack_limit: float,
    identifier=None,
    min_time_limit: float = 0.1,
    gt_time_limit: float = 2.0,
) -> Dict[str, Result]:
    ret = {
        "completion_id": completion_id,
        "task_id": problem["task_id"],
        "_identifier": identifier,
        "solution": solution,
    }
    ret["base"] = untrusted_check(
        solution,
        problem["test"],
        problem["entry_point"],
        max_as_limit,
        max_data_limit,
        max_stack_limit,
        min_time_limit,
        gt_time_limit,
    )
    return ret


def _plain(value):
    """Returns ``value`` with NumPy scalars and arrays turned into plain Python values."""
    def convert(item):
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, np.ndarray):
            return item.tolist()
        raise TypeError(f"Cannot serialize {type(item).__name__}")
    return json.loads(json.dumps(value, default=convert))


def evaluate(
    split: str,
    subset: str,
    samples: str,
    pass_k: str="1,5,10",
    parallel: int = -1,
    min_time_limit: float = 1,
    max_as_limit: int = 30 * 1024,
    max_data_limit: int = 30 * 1024,
    max_stack_limit: int = 10,
    calibrated: bool = True,
    check_gt_only: bool = False,
    no_gt: bool = False,
    selective_evaluate: str = "",
):
    started = time.time()
    passk = [int(k.strip()) for k in pass_k.split(',') if k.strip().isdigit()]
    if int(parallel) < 1:
        n_workers = max(1, multiprocessing.cpu_count() // 2)
    else:
        n_workers = int(parallel)
    max_as_limit, max_data_limit, max_stack_limit = int(max_as_limit), int(max_data_limit), int(max_stack_limit)

    if check_gt_only:
        samples = "__dummy__.jsonl"

    problems = get_bigcodebench(subset=subset)

    # Add selective evaluation logic
    if selective_evaluate:
        selected_ids = ["BigCodeBench/" + id for id in sorted(set(selective_evaluate.split(",")))]
        problems = {k: v for k, v in problems.items() if k in selected_ids}
        if not problems:
            raise ValueError(f"None of the provided task IDs {selected_ids} were found in the dataset")

    dataset_hash = get_bigcodebench_hash(subset=subset)

    if not no_gt:
        expected_time = get_groundtruth(n_workers, problems, dataset_hash, check_gt_only, max_as_limit, max_data_limit, max_stack_limit, min_time_limit)
    else:
        expected_time = {task_id: None for task_id in problems}

    gt_pass_rate = np.mean([1 if v is not None else 0 for k, v in expected_time.items() if k in problems])
    failed_tasks = [k for k, v in expected_time.items() if v is None and k in problems]

    pass_at_k = dict()
    results = {
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "eval": {},
    }

    n_samples = 0
    if not check_gt_only:

        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            futures = []
            completion_id = Counter()
            eval_results = defaultdict(list)  # task_id ->
            remainings = set()

            for sample in load_solutions(samples):
                task_id = sample["task_id"]

                if task_id not in problems:
                    continue
                solution = (
                    sample["solution"]
                    if "solution" in sample
                    else problems[task_id]["complete_prompt"] + sample["completion"]
                )
                if calibrated:
                    solution = problems[task_id]["code_prompt"] + "\n    pass\n" + solution
                remainings.add(sample["_identifier"])
                args = (
                    completion_id[task_id],
                    problems[task_id],
                    solution,
                    max_as_limit,
                    max_data_limit,
                    max_stack_limit,
                    sample["_identifier"],
                    min_time_limit,
                    expected_time[task_id] if expected_time[task_id] else 20
                )
                futures.append(executor.submit(check_correctness, *args))
                completion_id[task_id] += 1
                n_samples += 1

            assert n_samples == len(remainings), "Missing problems in unfinished"
            assert len(completion_id) == len(problems), "Missing problems in samples"

            for future in as_completed(futures):
                result = future.result()
                remainings.remove(result["_identifier"])
                eval_results[result["task_id"]].append(result)
                del future, result
                gc.collect()

        # sort the results for each problem by completion_id
        for task_id, task_results in eval_results.items():
            task_results.sort(key=lambda x: x["completion_id"])
            results["eval"][task_id] = []
            for res in task_results:
                stat, details = res["base"]
                results["eval"][task_id].append(
                    {
                        "task_id": task_id,
                        "solution": res["solution"],
                        "status": stat,
                        "details": details,
                    }
                )

        # Calculate pass@k.
        total = np.array([len(r) for k, r in results["eval"].items() if k in problems])
        base_correct = []

        for key, res in results["eval"].items():
            if key not in problems:
                continue
            bc = sum([r["status"] == PASS for r in res])
            base_correct.append(bc)

        base_correct = np.array(base_correct)

        pass_at_k.update({
            f"pass@{k}": estimate_pass_at_k(total, base_correct, k).mean()
            for k in passk
            if total.min() >= k
        })

        del problems, futures
        gc.collect()

    pass_at_k["model"] = os.path.basename(samples).split("--bigcodebench-")[0]
    pass_at_k["split"] = split
    pass_at_k["subset"] = subset
    pass_at_k["calibrated"] = calibrated
    pass_at_k["gt_pass_rate"] = gt_pass_rate
    pass_at_k["failed_tasks"] = failed_tasks

    mode = "ground truth check" if check_gt_only else f"{n_samples} samples"
    log.info("Request: subset=%s, %s, %.1fs", subset, mode, time.time() - started)
    return _plain(results), _plain(pass_at_k)


interface = gr.Interface(
    fn=evaluate,
    inputs=[
        gr.Dropdown(["complete", "instruct"], label="BigCodeBench Split"),
        gr.Dropdown(["full", "hard"], label="BigCodeBench Subset"),
        gr.File(label="Samples Path (.jsonl)"),
        gr.Textbox(label="Pass k Values (comma-separated)", value="1,5,10"),
        gr.Slider(-1, multiprocessing.cpu_count(), step=1, label="Parallel Workers", value=-1),
        gr.Slider(0.1, 10, step=0.1, label="Min Time Limit", value=1),
        gr.Slider(1, 100 * 1024, step=1024, label="Max AS Limit", value=30 * 1024),
        gr.Slider(1, 100 * 1024, step=1024, label="Max Data Limit", value=30 * 1024),
        gr.Slider(1, 100, step=1, label="Max Stack Limit", value=10),
        gr.Checkbox(label="Calibrated", value=True),
        gr.Checkbox(label="Check GT Only"),
        gr.Checkbox(label="No GT"),
        gr.Textbox(label="Selective Evaluated Task IDs (comma-separated, e.g. '0,1,2')", value=""),
    ],
    outputs=[
        gr.JSON(label="Results"),
        gr.JSON(label="Eval Results"),
    ],
    api_name="predict",
)
interface.queue(default_concurrency_limit=None)


def preload_gt(subsets, retries, retry_wait):
    """Computes (and caches) the ground-truth timings of each subset, with bounded retries."""
    for subset in subsets:
        for attempt in range(1, retries + 1):
            log.info("Ground-truth preload for '%s' (attempt %d/%d)...", subset, attempt, retries)
            try:
                _, info = evaluate(split="complete", subset=subset, samples="", check_gt_only=True)
            except Exception:
                traceback.print_exc()
                if attempt == retries:
                    log.error("PRELOAD FAILED for '%s' after %d attempts; see the errors above.", subset, retries)
                    sys.exit(1)
                time.sleep(retry_wait)
                continue
            log.info(
                "PRELOAD OK: subset=%s, gt_pass_rate=%.4f, %d tasks fail their ground truth: %s",
                subset, info["gt_pass_rate"], len(info["failed_tasks"]), info["failed_tasks"],
            )
            break


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--preload", default="hard", help="Comma-separated subsets to preload (hard,full)")
    parser.add_argument("--retries", type=int, default=3, help="Attempts per subset preload")
    parser.add_argument("--retry_wait", type=float, default=30.0, help="Seconds between preload attempts")
    args = parser.parse_args()

    preload_gt([s.strip() for s in args.preload.split(",") if s.strip()], args.retries, args.retry_wait)
    log.info("Starting the Gradio app; the public URL follows.")
    interface.launch(share=True, show_error=True)

## 5. Start the evaluator

Starts the app in the background as the unprivileged user, checks the ground truth of the Hard tasks, then prints the share URL. The app's log goes to `/content/evaluator.log`.

**The ground-truth result matters:** `gt_pass_rate` should be close to 1.0. Tasks whose reference solution fails here are environment failures, not model failures; EvaLoop excludes them from the run and records them. Many failures mean the environment is broken; don't start the main runs until that's fixed.

In [ ]:
import collections
import re
import subprocess
import time

subprocess.run(f"chmod -R a+rX /opt/bcb-app {ENV_DIR} {PYTHON_DIR}", shell=True, check=True)
subprocess.run(["pkill", "-f", APP_PATH])  # an evaluator left from an earlier run of this cell (no shell: see step 7)
# Ground-truth results are cached per dataset; clear them so a fixed environment is re-checked.
subprocess.run(f"rm -f /home/{RUN_USER}/.cache/bigcodebench/*.pkl", shell=True)
log_file = open(LOG_PATH, "w")
evaluator = subprocess.Popen(
    ["runuser", "-u", RUN_USER, "--", "env", "-i", *EVALUATOR_ENV,
     f"{ENV_DIR}/bin/python", APP_PATH, "--preload", "hard"],
    stdout=log_file, stderr=subprocess.STDOUT, cwd=f"/home/{RUN_USER}",
)

url, printed, deadline = None, 0, time.time() + 3 * 60 * 60
while url is None and evaluator.poll() is None and time.time() < deadline:
    time.sleep(10)
    lines = open(LOG_PATH).read().splitlines()
    for line in lines[printed:]:
        if "PRELOAD" in line or "Ground-truth preload" in line or "Error" in line or "Traceback" in line:
            print(line, flush=True)
    printed = len(lines)
    match = re.search(r"https://[a-z0-9-]+\.gradio\.live", "\n".join(lines))
    url = match.group(0) if match else None

if url is None:
    print(open(LOG_PATH).read()[-5000:])
    raise RuntimeError("The evaluator did not start; see the log above.")

# Why reference solutions failed (read from the log; nothing is executed here).
log = open(LOG_PATH).read()
rate = re.search(r"gt_pass_rate=([0-9.]+)", log)
missing = collections.Counter(re.findall(r"No module named .?'([A-Za-z0-9_.]+)", log))
print()
print("Missing modules in failing reference solutions:", missing.most_common() or "none")
if rate and float(rate.group(1)) < 0.95:
    print(f"WARNING: only {float(rate.group(1)):.0%} of reference solutions pass. Fix the environment before the main runs.")

print()
print("Evaluator URL (secret; anyone with it can run code on this VM until the runtime stops):")
print()
print("   ", url)
print()
print("In your terminal, without saving it to a file:")
print("    export EVALOOP_GRADIO_ENDPOINT='<the URL above>'")
print()
print("The share link lasts at most 1 week (Gradio 6.29.1, best effort) and ends when this runtime stops.")

## 6. Keep the runtime busy while EvaLoop runs

Prints the app's log as requests come in: one line per request, with the subset, sample count and duration. Leave it running for the whole EvaLoop run; Colab disconnects idle sessions sooner. Stopping this cell (■) stops only the log view, not the evaluator. If the stop button doesn't respond, restart the kernel: the evaluator keeps running, and step 7 still stops it.

In [ ]:
import time

printed = 0
try:
    while evaluator.poll() is None:
        lines = open(LOG_PATH).read().splitlines()
        for line in lines[printed:]:
            if "gradio.live" not in line:  # the URL was printed once in step 5
                print(line, flush=True)
        printed = len(lines)
        for _ in range(30):  # short sleeps, so the stop button (interrupt) is caught quickly
            time.sleep(1)
    print(f"The evaluator stopped (exit code {evaluator.returncode}).")
except KeyboardInterrupt:
    print("Log view stopped; the evaluator is still running.")

## If the session disconnects

Free-tier sessions are limited: Colab disconnects idle notebooks and ends sessions after a few hours at most (the exact limits vary and are not guaranteed). A disconnect deletes the VM, so the evaluator, its environment and its URL are gone.

EvaLoop checkpoints after every phase. When the evaluator becomes unreachable, the run stops with its progress saved and prints the resume command. To continue:

1. Start a fresh runtime and run this notebook again from step 1. You get a **new URL**.
2. In your terminal, set the new URL and resume with the same model and the checkpoint:

   ```
   export EVALOOP_GRADIO_ENDPOINT='<new URL>'
   evaloop evaluate --dataset bigcodebench --model <same model> \
     --resume_from results/<experiment_name>.checkpoint.json
   ```

The resume first health-checks the new evaluator, then continues without regenerating code it already has.


When EvaLoop has finished, delete the runtime so the Gradio URL stops working immediately.